# Synthetic Dataset Ground Truth Analysis

Analyzes the `correlated_noise` synthetic benchmark where feature identities are known exactly from `make_classification`.
Uses the **Broad** ground truth definition: informative + redundant features (indices 0..54), which is appropriate for
Boruta's *all-relevant* selection objective — redundant features are linearly correlated with the informative ones and
are legitimately relevant.

For **Vanilla Boruta** we get a full checkpoint trajectory (event-driven, logged when confirmed/rejected sets change).  
For **GreedyParallel** we get a single final-state snapshot per run.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='paper', font_scale=1.2)
plt.rcParams['figure.dpi'] = 110

## 1. Load Data

In [ ]:
GT_FILE   = 'data/benchmark_gt_results_vGT.csv'
MAIN_FILE = 'data/benchmark_results_vGT.csv'
TARGET_ESTIMATOR = 'extra_trees'
DATASET = 'correlated_noise'
GT = 'Broad'   # Single GT definition used throughout

df_gt   = pd.read_csv(GT_FILE)
df_main = pd.read_csv(MAIN_FILE)

df_gt   = df_gt[(df_gt['Dataset'] == DATASET) & (df_gt['Estimator'] == TARGET_ESTIMATOR)].copy()
df_main = df_main[(df_main['Dataset'] == DATASET) & (df_main['Estimator'] == TARGET_ESTIMATOR)].copy()

DATASET = 'Synthetic Dataset'

print(f'GT log rows  : {len(df_gt)}')
print(f'Main log rows: {len(df_main)}')
print(f'Methods      : {df_gt["Method"].unique()}')
print(f'Alpha values : {sorted(df_gt["Alpha"].unique())}')
print(f'GT Broad size (should be 55): {df_gt["GT_Broad_Size"].iloc[0]}')
df_gt.head()

## 2. Data Preparation

In [ ]:
df_vanilla = df_gt[df_gt['Method'] == 'Vanilla'].copy()
df_greedy  = df_gt[df_gt['Method'].str.contains('Greedy')].copy()

if 'GreedyParallel' in df_greedy['Method'].values:
    df_greedy = df_greedy[df_greedy['Method'] == 'GreedyParallel'].copy()
    greedy_label = 'GreedyParallel'
else:
    greedy_label = 'Greedy'

df_vanilla_final = df_vanilla[df_vanilla['Is_Final'] == True].copy()
df_greedy_final  = df_greedy.copy()

alphas     = sorted(df_gt['Alpha'].unique())
n_features = int(df_gt['n_features_orig'].iloc[0])
gt_broad   = int(df_gt['GT_Broad_Size'].iloc[0])

print(f'Vanilla checkpoints      : {len(df_vanilla)}')
print(f'Vanilla final-state rows : {len(df_vanilla_final)}')
print(f'{greedy_label} snapshots : {len(df_greedy)}')
print(f'Alpha values: {alphas}')
print(f'n_features={n_features}, GT broad={gt_broad}')

## 3. Palette & Helpers

In [ ]:
PAL_METHOD = {
    'Vanilla'       : '#3498db',
    'GreedyParallel': '#00b894',
    'Greedy'        : '#2ecc71',
}

# Marker for Greedy: filled pentagon renders cleanly at all sizes
GREEDY_MARKER = 'x'
VANILLA_MARKER = 'o'

METRICS = ['Recall', 'Precision', 'Specificity', 'F1']

def gt_col(metric):
    return f'{metric}_{GT}'

def greedy_hline(ax, metric, alpha, idx=0):
    """Add a dashed horizontal reference line for the Greedy mean."""
    g = df_greedy[df_greedy['Alpha'] == alpha][gt_col(metric)]
    if len(g) == 0:
        return
    gm = g.mean()
    ax.axhline(
        gm,
        color=PAL_METHOD.get(greedy_label, '#00b894'),
        linestyle='--', linewidth=2,
        label=f'{greedy_label} mean' if idx == 0 else ''
    )
    return gm

In [ ]:
# ── Global single-alpha selector ─────────────────────────────────────────────
# Change this value to focus all single-alpha plots on a different alpha.
FOCUS_ALPHA = 0.01
# ─────────────────────────────────────────────────────────────────────────────

## 4. Final-State Summary Table

In [ ]:
final_rows = []
for metric in METRICS:
    col = gt_col(metric)
    for alpha in alphas:
        for method_df, method_name in [
            (df_vanilla_final, 'Vanilla'),
            (df_greedy_final,  greedy_label)
        ]:
            subset = method_df[method_df['Alpha'] == alpha][col]
            if len(subset) == 0:
                continue
            final_rows.append({
                'Method': method_name,
                'Alpha' : str(alpha),
                'Metric': metric,
                'Mean'  : subset.mean(),
                'Std'   : subset.std(),
            })

df_final_summary = pd.DataFrame(final_rows)

print(f'Final-state metrics — Broad GT (mean ± std across runs):')
pivot = df_final_summary.pivot_table(
    index=['Method', 'Alpha'],
    columns='Metric',
    values='Mean'
)[METRICS].round(3)
display(pivot)

## 5. Final-State Barplot: All Metrics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

method_order  = ['Vanilla', greedy_label]
method_colors = [PAL_METHOD['Vanilla'], PAL_METHOD.get(greedy_label, '#00b894')]
alpha_order   = [str(a) for a in sorted(alphas)]   # numeric sort → string labels

for i, metric in enumerate(METRICS):
    ax = axes[i]
    col = gt_col(metric)

    plot_data = pd.concat([
        df_vanilla_final[['Method', 'Alpha', col]],
        df_greedy_final[['Method',  'Alpha', col]]
    ], ignore_index=True)
    plot_data['Alpha'] = plot_data['Alpha'].astype(str)

    # --- strip plot (all individual points) ---
    sns.stripplot(
        data=plot_data, x='Alpha', y=col, hue='Method',
        hue_order=method_order,
        palette=dict(zip(method_order, method_colors)),
        order=alpha_order,
        ax=ax,
        dodge=True, jitter=0.15,
        size=6, alpha=0.65, linewidth=0.4, edgecolor='white',
        zorder=3,
    )

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('Alpha', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    # Zoom into the region where the data actually lives
    col_min = plot_data[col].min()
    ax.set_ylim(max(0, col_min - 0.05), 1.02)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

    # Build a clean legend: strip entries + mean marker entries
    handles, labels = ax.get_legend_handles_labels()
    # stripplot adds one handle per method; mean scatter adds two more → deduplicate
    seen, clean_h, clean_l = set(), [], []
    for h, l in zip(handles, labels):
        if l not in seen:
            seen.add(l); clean_h.append(h); clean_l.append(l)
    ax.legend(clean_h, clean_l, title='Method', fontsize=9, title_fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(
    f'Final-State Ground Truth Metrics (Broad GT)\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 6. Confusion Breakdown at Final State

TP / FP / FN / TN counts show *how* selections are wrong — false alarms vs misses.

In [ ]:
count_rows = []
for alpha in alphas:
    for method_df, method_name in [
        (df_vanilla_final, 'Vanilla'),
        (df_greedy_final,  greedy_label)
    ]:
        sub = method_df[method_df['Alpha'] == alpha]
        if len(sub) == 0:
            continue
        for cnt in ['TP', 'FP', 'FN', 'TN']:
            count_rows.append({
                'Method'    : method_name,
                'Alpha'     : str(alpha),
                'Count_Type': cnt,
                'Count'     : sub[f'{cnt}_{GT}'].mean()
            })

df_counts = pd.DataFrame(count_rows)
cnt_pal   = {'TP': '#2ecc71', 'FP': '#e74c3c', 'FN': '#e67e22', 'TN': '#95a5a6'}

fig, axes = plt.subplots(1, len(alphas), figsize=(5 * len(alphas), 5), sharey=True)
if len(alphas) == 1:
    axes = [axes]

for idx, alpha in enumerate(alphas):
    ax = axes[idx]
    sub = df_counts[df_counts['Alpha'] == str(alpha)]
    sns.barplot(
        data=sub, x='Method', y='Count', hue='Count_Type',
        hue_order=['TP', 'FP', 'FN', 'TN'],
        palette=cnt_pal, ax=ax
    )
    ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Method', fontsize=11)
    ax.set_ylabel('Mean Count (across runs)' if idx == 0 else '', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(title='Type', fontsize=9)

fig.suptitle(
    f'Confusion Breakdown — Broad GT\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## 7. Convergence Trajectories vs Iteration (Vanilla)

Each line is one run. Dashed line = GreedyParallel final mean.

In [ ]:
def greedy_iterations(alpha, n_features, bonferroni=True):
    """Replicates GreedyBorutaParallel._compute_iterations_to_convergence."""
    threshold = alpha / float(n_features) if bonferroni else alpha
    return int(np.ceil(np.log2(1.0 / threshold)))


def plot_metric_trajectories(metric, x_col='Iteration', x_label='Iteration'):
    col = gt_col(metric)
    n_alphas = len(alphas)
    fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
    axes = axes.flatten()

    for idx, alpha in enumerate(alphas):
        ax = axes[idx]
        cp = df_vanilla[df_vanilla['Alpha'] == alpha]

        for run_id in sorted(cp['Run_ID'].unique()):
            run = cp[cp['Run_ID'] == run_id].sort_values('Checkpoint_ID')
            ax.plot(
                run[x_col], run[col],
                marker=VANILLA_MARKER, markersize=5, linewidth=2, alpha=0.65,
                color=PAL_METHOD['Vanilla'],
                label='Vanilla runs' if (idx == 0 and run_id == sorted(cp['Run_ID'].unique())[0]) else ''
            )

        # GreedyParallel reference (horizontal dashed line)
        g_vals = df_greedy[df_greedy['Alpha'] == alpha][col]
        if len(g_vals):
            ax.axhline(
                g_vals.mean(),
                color=PAL_METHOD.get(greedy_label, '#00b894'),
                linestyle='--', linewidth=2.5,
                label=f'{greedy_label} mean' if idx == 0 else ''
            )

        # Vertical line: GreedyBoruta reference on the x-axis
        if x_col == 'Iteration':
            greedy_x = greedy_iterations(alpha, n_features)
            vline_label = f'{greedy_label} iters ({greedy_x})' if idx == 0 else ''
        else:
            g_alpha = df_greedy[df_greedy['Alpha'] == alpha]
            greedy_x = g_alpha['Elapsed_Sec'].mean() if len(g_alpha) else None
            vline_label = f'{greedy_label} mean time' if idx == 0 else ''

        if greedy_x is not None:
            ax.axvline(
                greedy_x,
                color=PAL_METHOD.get(greedy_label, '#00b894'),
                linestyle=':', linewidth=1.8, alpha=0.7,
                label=vline_label
            )

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel(f'{metric} (Broad)', fontsize=11)
        ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
        ax.set_ylim(-0.02, 1.05)
        ax.grid(True, alpha=0.3)
        if idx == 0:
            ax.legend(fontsize=9)

    fig.suptitle(
        f'{metric} (Broad GT) vs {x_label}\n{DATASET} | {TARGET_ESTIMATOR}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

for metric in METRICS:
    plot_metric_trajectories(metric, x_col='Iteration', x_label='Iteration')

## 7b. Convergence Trajectories — Single Alpha (Iteration)

All four metrics in one figure for `FOCUS_ALPHA`. Each line is one Vanilla run; dashed = GreedyParallel mean.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cp_focus = df_vanilla[df_vanilla['Alpha'] == FOCUS_ALPHA]
greedy_x = greedy_iterations(FOCUS_ALPHA, n_features)

for i, metric in enumerate(METRICS):
    ax = axes[i]
    col = gt_col(metric)

    for run_id in sorted(cp_focus['Run_ID'].unique()):
        run = cp_focus[cp_focus['Run_ID'] == run_id].sort_values('Checkpoint_ID')
        ax.plot(
            run['Iteration'], run[col],
            marker=VANILLA_MARKER, markersize=4, linewidth=1.8, alpha=0.6,
            color=PAL_METHOD['Vanilla'],
            label='Vanilla runs' if run_id == sorted(cp_focus['Run_ID'].unique())[0] else ''
        )

    g_vals = df_greedy[df_greedy['Alpha'] == FOCUS_ALPHA][col]
    if len(g_vals):
        ax.axhline(g_vals.mean(),
                   color=PAL_METHOD.get(greedy_label, '#00b894'),
                   linestyle='--', linewidth=2.5, label=f'{greedy_label} mean')

    ax.axvline(greedy_x,
               color=PAL_METHOD.get(greedy_label, '#00b894'),
               linestyle=':', linewidth=1.8, alpha=0.7,
               label=f'{greedy_label} iters ({greedy_x})')

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel(f'{metric} (Broad)', fontsize=11)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle(
    f'Convergence Trajectories vs Iteration — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 8. Convergence Trajectories vs Elapsed Time

In [ ]:
for metric in ['Recall', 'Precision', 'F1']:
    plot_metric_trajectories(metric, x_col='Elapsed_Sec', x_label='Time (seconds)')

## 8b. Convergence Trajectories — Single Alpha (Time)

All four metrics for `FOCUS_ALPHA`, x-axis in elapsed seconds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

cp_focus = df_vanilla[df_vanilla['Alpha'] == FOCUS_ALPHA]
g_focus  = df_greedy[df_greedy['Alpha'] == FOCUS_ALPHA]
greedy_time = g_focus['Elapsed_Sec'].mean() if len(g_focus) else None

for i, metric in enumerate(METRICS):
    ax = axes[i]
    col = gt_col(metric)

    for run_id in sorted(cp_focus['Run_ID'].unique()):
        run = cp_focus[cp_focus['Run_ID'] == run_id].sort_values('Checkpoint_ID')
        ax.plot(
            run['Elapsed_Sec'], run[col],
            marker=VANILLA_MARKER, markersize=4, linewidth=1.8, alpha=0.6,
            color=PAL_METHOD['Vanilla'],
            label='Vanilla runs' if run_id == sorted(cp_focus['Run_ID'].unique())[0] else ''
        )

    g_vals = g_focus[col]
    if len(g_vals):
        ax.axhline(g_vals.mean(),
                   color=PAL_METHOD.get(greedy_label, '#00b894'),
                   linestyle='--', linewidth=2.5, label=f'{greedy_label} mean')

    if greedy_time is not None:
        ax.axvline(greedy_time,
                   color=PAL_METHOD.get(greedy_label, '#00b894'),
                   linestyle=':', linewidth=1.8, alpha=0.7,
                   label=f'{greedy_label} mean time')

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel(f'{metric} (Broad)', fontsize=11)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle(
    f'Convergence Trajectories vs Time — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 9. Recall–Precision Path (Broad GT)

Phase-space view of each Vanilla run's trajectory. Square = start, star = end.  
Pentagon = GreedyParallel final mean. Ideal target is the top-right corner (1, 1).

In [ ]:
n_alphas = len(alphas)
fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
axes = axes.flatten()

for idx, alpha in enumerate(alphas):
    ax = axes[idx]
    cp = df_vanilla[df_vanilla['Alpha'] == alpha]

    for run_id in sorted(cp['Run_ID'].unique()):
        run = cp[cp['Run_ID'] == run_id].sort_values('Checkpoint_ID')
        ax.plot(
            run[gt_col('Recall')], run[gt_col('Precision')],
            marker=VANILLA_MARKER, markersize=4, linewidth=1.8, alpha=0.6,
            color=PAL_METHOD['Vanilla']
        )
        # Start marker (square)
        ax.scatter(
            run[gt_col('Recall')].iloc[0], run[gt_col('Precision')].iloc[0],
            color='#2c3e50', zorder=6, s=55, marker='s',
            label='Start' if run_id == sorted(cp['Run_ID'].unique())[0] else ''
        )
        # End marker (star)
        ax.scatter(
            run[gt_col('Recall')].iloc[-1], run[gt_col('Precision')].iloc[-1],
            color='#2c3e50', zorder=6, s=80, marker='*',
            label='End' if run_id == sorted(cp['Run_ID'].unique())[0] else ''
        )

    # GreedyParallel reference point — filled pentagon
    g = df_greedy[df_greedy['Alpha'] == alpha]
    if len(g):
        ax.scatter(
            g[gt_col('Recall')].mean(), g[gt_col('Precision')].mean(),
            color=PAL_METHOD.get(greedy_label, '#00b894'),
            s=250, marker=GREEDY_MARKER, zorder=7,
            edgecolors='black', linewidths=0.8,
            label=greedy_label
        )

    ax.set_xlabel('Recall (Broad)', fontsize=11)
    ax.set_ylabel('Precision (Broad)', fontsize=11)
    ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
    ax.set_xlim(-0.02, 1.05)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.3)
    ax.annotate(
        'Ideal\n(1, 1)', xy=(1, 1), xytext=(0.78, 0.86),
        fontsize=9, color='grey',
        arrowprops=dict(arrowstyle='->', color='grey', lw=1.2)
    )
    ax.legend(fontsize=9, loc='lower left')

fig.suptitle(
    f'Recall–Precision Path (Broad GT) — Vanilla Boruta\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## 9b. Recall–Precision Path — Single Alpha

Phase-space trajectories for `FOCUS_ALPHA` only.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

cp_focus = df_vanilla[df_vanilla['Alpha'] == FOCUS_ALPHA]
run_ids  = sorted(cp_focus['Run_ID'].unique())

for run_id in run_ids:
    run = cp_focus[cp_focus['Run_ID'] == run_id].sort_values('Checkpoint_ID')
    ax.plot(
        run[gt_col('Recall')], run[gt_col('Precision')],
        marker=VANILLA_MARKER, markersize=4, linewidth=1.8, alpha=0.6,
        color=PAL_METHOD['Vanilla']
    )
    ax.scatter(
        run[gt_col('Recall')].iloc[0], run[gt_col('Precision')].iloc[0],
        color='#2c3e50', zorder=6, s=55, marker='s',
        label='Start' if run_id == run_ids[0] else ''
    )
    ax.scatter(
        run[gt_col('Recall')].iloc[-1], run[gt_col('Precision')].iloc[-1],
        color='#2c3e50', zorder=6, s=80, marker='*',
        label='End' if run_id == run_ids[0] else ''
    )

# Vanilla legend proxy
ax.plot([], [], color=PAL_METHOD['Vanilla'], linewidth=2, label='Vanilla runs')

# GreedyParallel reference point
g = df_greedy[df_greedy['Alpha'] == FOCUS_ALPHA]
if len(g):
    ax.scatter(
        g[gt_col('Recall')].mean(), g[gt_col('Precision')].mean(),
        color=PAL_METHOD.get(greedy_label, '#00b894'),
        s=250, marker=GREEDY_MARKER, zorder=7,
        edgecolors='black', linewidths=0.8,
        label=greedy_label
    )

ax.set_xlabel('Recall (Broad)', fontsize=12)
ax.set_ylabel('Precision (Broad)', fontsize=12)
ax.set_xlim(-0.02, 1.05)
ax.set_ylim(-0.02, 1.05)
ax.grid(True, alpha=0.3)
# ax.annotate(
#     'Ideal\n(1, 1)', xy=(1, 1), xytext=(0.78, 0.86),
#     fontsize=9, color='grey',
#     arrowprops=dict(arrowstyle='->', color='grey', lw=1.2)
# )
ax.annotate(
    'Ideal\n(1, 1)', xy=(1, 1), xytext=(1, 0.9),
    fontsize=9, color='grey',
    arrowprops=dict(arrowstyle='->', color='grey', lw=1.2)
)
ax.legend(fontsize=10, loc='lower left')
ax.set_title(
    f'Recall–Precision Path (Broad GT) — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

## 10. Feature Count Trajectories (Confirmed / Tentative / Rejected)

In [ ]:
count_pal = {
    'N_Confirmed': '#2ecc71',
    'N_Tentative': '#3498db',
    'N_Rejected' : '#e74c3c'
}

n_alphas = len(alphas)
fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
axes = axes.flatten()

for idx, alpha in enumerate(alphas):
    ax = axes[idx]
    cp = df_vanilla[df_vanilla['Alpha'] == alpha]
    agg = cp.groupby('Iteration')[list(count_pal)].mean()

    for col, color in count_pal.items():
        ax.plot(
            agg.index, agg[col],
            marker=VANILLA_MARKER, markersize=4, linewidth=2,
            color=color, label=col.replace('N_', '')
        )

    # Mark GT Broad size
    ax.axhline(gt_broad, color='grey', linestyle=':', linewidth=1.5,
               label=f'GT size ({gt_broad})')

    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel('Mean Feature Count' if idx == 0 else '', fontsize=11)
    ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle(
    f'Feature State Counts over Iterations — Vanilla\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## 10b. Feature Count Trajectories — Single Alpha

Confirmed / Tentative / Rejected mean counts for `FOCUS_ALPHA`.

In [ ]:
count_pal = {
    'N_Confirmed': '#2ecc71',
    'N_Tentative': '#3498db',
    'N_Rejected' : '#e74c3c'
}

fig, ax = plt.subplots(figsize=(9, 5))

cp_focus = df_vanilla[(df_vanilla['Alpha'] == FOCUS_ALPHA)  & (df_vanilla['Run_ID'] == 0)]
agg = cp_focus.groupby('Iteration')[list(count_pal)].mean()

for col, color in count_pal.items():
    ax.plot(
        agg.index, agg[col],
        marker=VANILLA_MARKER, markersize=4, linewidth=2,
        color=color, label=col.replace('N_', '')
    )

ax.axhline(gt_broad, color='grey', linestyle=':', linewidth=1.5,
           label=f'GT size ({gt_broad})')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Mean Feature Count', fontsize=12)
ax.set_title(
    f'Feature State Counts over Iterations — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=13, fontweight='bold'
)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 11b. Undecidedness Factor & Potential Miss Rate (Vanilla)

In [ ]:
def plot_undecidedness_factor(x_axis='iteration'):
    """
    Plot the undecidedness factor (N_Tentative / all_features)
    for Vanilla Boruta across all alpha values.

    Parameters
    ----------
    x_axis : str
        'iteration' or 'time'
    """
    x_col   = 'Elapsed_Sec' if x_axis == 'time' else 'Iteration'
    x_label = 'Time (seconds)' if x_axis == 'time' else 'Iteration'

    n_alphas = len(alphas)
    fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
    axes = axes.flatten()

    for idx, alpha in enumerate(alphas):
        ax = axes[idx]
        cp = df_vanilla[df_vanilla['Alpha'] == alpha].copy()

        all_features = cp['N_Tentative'] + cp['N_Rejected'] + cp['N_Confirmed']
        cp['Undecidedness_Factor'] = np.where(
            all_features > 0,
            cp['N_Tentative'] / all_features,
            np.nan
        )

        for run_id in sorted(cp['Run_ID'].unique()):
            run = cp[(cp['Run_ID'] == run_id) & (cp['Checkpoint_ID'] > 0)].sort_values('Checkpoint_ID')
            ax.plot(
                run[x_col], run['Undecidedness_Factor'],
                marker=VANILLA_MARKER, markersize=5, linewidth=2, alpha=0.6,
                color=PAL_METHOD['Vanilla'],
                label='Vanilla run' if (idx == 0 and run_id == sorted(cp['Run_ID'].unique())[0]) else ''
            )

        ax.axhline(0, color='green', linestyle='--', linewidth=1.5, alpha=0.6,
                   label='All Decided' if idx == 0 else '')

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel('Undecidedness Factor\n(Tentative / All Features)', fontsize=11)
        ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
        ax.set_ylim(bottom=-0.01)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax.grid(True, alpha=0.3)
        if idx == 0:
            ax.legend(fontsize=9)

    x_type = 'Time' if x_axis == 'time' else 'Iteration'
    fig.suptitle(
        f'Undecidedness Factor vs {x_type} — Vanilla\n{DATASET} | {TARGET_ESTIMATOR}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()


def plot_potential_miss_rate(x_axis='iteration'):
    """
    Plot the potential miss rate (N_Tentative / (N_Tentative + N_Confirmed))
    for Vanilla Boruta across all alpha values.

    Parameters
    ----------
    x_axis : str
        'iteration' or 'time'
    """
    x_col   = 'Elapsed_Sec' if x_axis == 'time' else 'Iteration'
    x_label = 'Time (seconds)' if x_axis == 'time' else 'Iteration'

    n_alphas = len(alphas)
    fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
    axes = axes.flatten()

    for idx, alpha in enumerate(alphas):
        ax = axes[idx]
        cp = df_vanilla[df_vanilla['Alpha'] == alpha].copy()

        denom = cp['N_Tentative'] + cp['N_Confirmed']
        cp['Potential_Miss_Rate'] = np.where(
            denom > 0,
            cp['N_Tentative'] / denom,
            np.nan
        )

        for run_id in sorted(cp['Run_ID'].unique()):
            run = cp[(cp['Run_ID'] == run_id) & (cp['Checkpoint_ID'] > 0)].sort_values('Checkpoint_ID')
            run_valid = run[~run['Potential_Miss_Rate'].isna()]
            if len(run_valid) == 0:
                continue
            ax.plot(
                run_valid[x_col], run_valid['Potential_Miss_Rate'],
                marker=VANILLA_MARKER, markersize=5, linewidth=2, alpha=0.6,
                color=PAL_METHOD['Vanilla'],
                label='Vanilla run' if (idx == 0 and run_id == sorted(cp['Run_ID'].unique())[0]) else ''
            )

        ax.axhline(0, color='green', linestyle='--', linewidth=1.5, alpha=0.6,
                   label='All Confirmed' if idx == 0 else '')

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel('Potential Miss Rate\n(Tentative / (Tentative + Confirmed))', fontsize=11)
        ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
        ax.set_ylim(bottom=-0.01)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax.grid(True, alpha=0.3)
        if idx == 0:
            ax.legend(fontsize=9)

    x_type = 'Time' if x_axis == 'time' else 'Iteration'
    fig.suptitle(
        f'Potential Miss Rate vs {x_type} — Vanilla\n{DATASET} | {TARGET_ESTIMATOR}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()


print('Undecidedness Factor vs Iteration:')
plot_undecidedness_factor(x_axis='iteration')

print('Potential Miss Rate vs Iteration:')
plot_potential_miss_rate(x_axis='iteration')

print('Potential Miss Rate vs Time:')
plot_potential_miss_rate(x_axis='time')

## 11c. Undecidedness & Potential Miss Rate — Single Alpha

Both diagnostic metrics for `FOCUS_ALPHA` side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cp_focus = df_vanilla[df_vanilla['Alpha'] == FOCUS_ALPHA].copy()

all_features = cp_focus['N_Tentative'] + cp_focus['N_Rejected'] + cp_focus['N_Confirmed']
cp_focus['Undecidedness_Factor'] = np.where(all_features > 0, cp_focus['N_Tentative'] / all_features, np.nan)

denom = cp_focus['N_Tentative'] + cp_focus['N_Confirmed']
cp_focus['Potential_Miss_Rate'] = np.where(denom > 0, cp_focus['N_Tentative'] / denom, np.nan)

run_ids = sorted(cp_focus['Run_ID'].unique())

for col_name, ax, ylabel, title in [
    ('Undecidedness_Factor', axes[0],
     'Undecidedness Factor\n(Tentative / All Features)', 'Undecidedness Factor'),
    ('Potential_Miss_Rate',  axes[1],
     'Potential Miss Rate\n(Tentative / (Tentative + Confirmed))', 'Potential Miss Rate'),
]:
    for run_id in run_ids:
        run = cp_focus[(cp_focus['Run_ID'] == run_id) & (cp_focus['Checkpoint_ID'] > 0)].sort_values('Checkpoint_ID')
        run_valid = run[~run[col_name].isna()]
        if len(run_valid) == 0:
            continue
        ax.plot(
            run_valid['Iteration'], run_valid[col_name],
            marker=VANILLA_MARKER, markersize=5, linewidth=2, alpha=0.6,
            color=PAL_METHOD['Vanilla'],
            label='Vanilla run' if run_id == run_ids[0] else ''
        )

    ax.axhline(0, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Fully resolved')
    ax.set_xlabel('Iteration', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=-0.01)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle(
    f'Undecidedness & Potential Miss Rate — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

## 11. Heatmap: Final F1 (Broad) by Method × Alpha

In [ ]:
col = gt_col('F1')

both = pd.concat([
    df_vanilla_final[['Method', 'Alpha', col]],
    df_greedy_final[['Method',  'Alpha', col]]
], ignore_index=True)

pivot = both.groupby(['Method', 'Alpha'])[col].mean().unstack('Alpha')

fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='YlGn',
    vmin=0.9, vmax=1.0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'F1 (Broad)'}
)
ax.set_title(
    f'Mean Final F1 (Broad GT) — Method × Alpha\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Alpha', fontsize=11)
ax.set_ylabel('Method', fontsize=11)
plt.tight_layout()
plt.show()

## 12. Speed vs GT Quality Trade-off

Runtime vs final F1 (Broad). Point size ∝ N_Confirmed. Pentagon = GreedyParallel.

In [ ]:
col = gt_col('F1')

runtime_cols = ['Dataset', 'Method', 'Alpha', 'Run_ID', 'Time_Sec']
df_rt = df_main[runtime_cols].copy()
df_rt['Alpha'] = df_rt['Alpha'].astype(str)

fig, ax = plt.subplots(figsize=(9, 6))

for method_df, method_name, marker, msize in [
    (df_vanilla_final.copy(), 'Vanilla',    VANILLA_MARKER, 1),
    (df_greedy_final.copy(),  greedy_label, GREEDY_MARKER,  1)
]:
    method_df = method_df.copy()
    method_df['Alpha'] = method_df['Alpha'].astype(str)
    merged = method_df.merge(
        df_rt[df_rt['Method'] == method_name],
        on=['Dataset', 'Method', 'Alpha', 'Run_ID'],
        how='left'
    )
    agg = merged.groupby('Alpha').agg(
        F1_mean   =(col,        'mean'),
        Time_mean =('Time_Sec', 'mean'),
        N_Conf_mean=('N_Confirmed', 'mean')
    ).reset_index()

    ax.scatter(
        agg['Time_mean'], agg['F1_mean'],
        s=agg['N_Conf_mean'] * 0.5 + msize,
        color=PAL_METHOD.get(method_name, '#95a5a6'),
        marker=marker, alpha=0.88,
        edgecolors='black', linewidths=0.7,
        label=method_name, zorder=5
    )
    for _, row in agg.iterrows():
        ax.annotate(
            f'α={row["Alpha"]}',
            (row['Time_mean'], row['F1_mean']),
            textcoords='offset points', xytext=(6, 4), fontsize=9
        )

ax.set_xlabel('Mean Runtime (seconds)', fontsize=12)
ax.set_ylabel('Mean Final F1 (Broad GT)', fontsize=12)
ax.set_title(
    f'Speed vs GT Quality Trade-off (Broad GT)\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Effect of Alpha — Boxplots Across Runs

In [ ]:
both_final = pd.concat([
    df_vanilla_final[['Method', 'Alpha'] + [gt_col(m) for m in METRICS] + ['N_Confirmed']],
    df_greedy_final[ ['Method', 'Alpha'] + [gt_col(m) for m in METRICS] + ['N_Confirmed']]
], ignore_index=True)
both_final['Alpha'] = both_final['Alpha'].astype(str)
alpha_order = [str(a) for a in sorted(alphas)]   # numeric sort

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, metric in enumerate(METRICS):
    ax = axes[i]
    col = gt_col(metric)
    sns.boxplot(
        data=both_final, x='Alpha', y=col, hue='Method',
        hue_order=['Vanilla', greedy_label],
        order=alpha_order,
        palette=PAL_METHOD, ax=ax, linewidth=1.2
    )
    ax.set_title(f'{metric} (Broad)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Alpha', fontsize=10)
    ax.set_ylabel(f'{metric} (Broad)', fontsize=10)
    ax.set_ylim(max(0, both_final[col].min() - 0.05), 1.05)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(title='Method', fontsize=8, title_fontsize=8)

fig.suptitle(
    f'Broad GT Metrics by Alpha — Method Comparison\n{DATASET} | {TARGET_ESTIMATOR}',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

## 14. Statistical Test: Greedy vs Vanilla (Broad GT)

Welch t-test on final-state metrics across runs.

In [ ]:
test_rows = []

for metric in METRICS:
    col = gt_col(metric)
    for alpha in alphas:
        v_scores = df_vanilla_final[df_vanilla_final['Alpha'] == alpha][col].values
        g_scores = df_greedy_final[ df_greedy_final['Alpha']  == alpha][col].values

        if len(v_scores) < 2 or len(g_scores) < 2:
            continue

        _, p_two          = stats.ttest_ind(v_scores, g_scores, equal_var=False)
        _, p_greedy_better = stats.ttest_ind(g_scores, v_scores, equal_var=False, alternative='greater')

        winner = 'Tie'
        if p_greedy_better < 0.05:
            winner = greedy_label
        elif p_two < 0.05 and v_scores.mean() > g_scores.mean():
            winner = 'Vanilla'

        test_rows.append({
            'Metric'          : metric,
            'Alpha'           : str(alpha),
            'Vanilla_Mean'    : v_scores.mean(),
            'Greedy_Mean'     : g_scores.mean(),
            'Diff (V-G)'      : v_scores.mean() - g_scores.mean(),
            'P_TwoSided'      : p_two,
            'P_Greedy_Better' : p_greedy_better,
            'Winner'          : winner
        })

df_tests = pd.DataFrame(test_rows)
print(f'Statistical comparison — Broad GT (significance = 0.05)')
print('-' * 95)
display(df_tests.round(4).to_string(index=False))

## 15. Mean Trajectory with Confidence Band (Vanilla)

Mean ± 1 std across runs. GreedyParallel shown as dashed reference ± shaded band.

In [ ]:
def plot_mean_trajectory_band(metric, x_col='Iteration', x_label='Iteration'):
    col = gt_col(metric)
    n_alphas = len(alphas)
    fig, axes = plt.subplots(1, n_alphas, figsize=(6 * n_alphas, 5), squeeze=False)
    axes = axes.flatten()

    for idx, alpha in enumerate(alphas):
        ax = axes[idx]
        cp = df_vanilla[df_vanilla['Alpha'] == alpha]

        agg = cp.groupby(x_col)[col].agg(['mean', 'std']).reset_index()
        agg['std'] = agg['std'].fillna(0)

        ax.plot(agg[x_col], agg['mean'],
                color=PAL_METHOD['Vanilla'], linewidth=2.5, label='Vanilla mean')
        ax.fill_between(
            agg[x_col],
            (agg['mean'] - agg['std']).clip(lower=0),
            (agg['mean'] + agg['std']).clip(upper=1),
            alpha=0.2, color=PAL_METHOD['Vanilla']
        )

        g_vals = df_greedy[df_greedy['Alpha'] == alpha][col]
        if len(g_vals):
            gm   = g_vals.mean()
            gstd = g_vals.std()
            ax.axhline(
                gm,
                color=PAL_METHOD.get(greedy_label, '#00b894'),
                linestyle='--', linewidth=2.2,
                label=f'{greedy_label} mean'
            )
            ax.axhspan(
                max(0, gm - gstd), min(1, gm + gstd),
                alpha=0.12, color=PAL_METHOD.get(greedy_label, '#00b894')
            )

        # Vertical line: GreedyBoruta reference on the x-axis
        if x_col == 'Iteration':
            greedy_x = greedy_iterations(alpha, n_features)
            vline_label = f'{greedy_label} iters ({greedy_x})'
        else:
            g_alpha = df_greedy[df_greedy['Alpha'] == alpha]
            greedy_x = g_alpha['Elapsed_Sec'].mean() if len(g_alpha) else None
            vline_label = f'{greedy_label} mean time'

        if greedy_x is not None:
            ax.axvline(
                greedy_x,
                color=PAL_METHOD.get(greedy_label, '#00b894'),
                linestyle=':', linewidth=1.8, alpha=0.7,
                label=vline_label
            )

        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel(f'{metric} (Broad)', fontsize=11)
        ax.set_title(f'Alpha = {alpha}', fontsize=12, fontweight='bold')
        ax.set_ylim(-0.02, 1.05)
        ax.grid(True, alpha=0.3)
        if idx == 0:
            ax.legend(fontsize=9)

    fig.suptitle(
        f'Mean {metric} ± 1 Std (Broad GT) vs {x_label}\n{DATASET} | {TARGET_ESTIMATOR}',
        fontsize=14, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()

for metric in ['Recall', 'F1']:
    plot_mean_trajectory_band(metric, x_col='Iteration',   x_label='Iteration')
    plot_mean_trajectory_band(metric, x_col='Elapsed_Sec', x_label='Time (seconds)')

## 15b. Mean Trajectory ± Band — Single Alpha

All four metrics for `FOCUS_ALPHA` in a 2×2 grid, both vs Iteration and vs Time.

In [ ]:
def _band_axes(ax, x_col, metric, cp_focus, g_focus, greedy_ref_x, x_label):
    col = gt_col(metric)
    agg = cp_focus.groupby(x_col)[col].agg(['mean', 'std']).reset_index()
    agg['std'] = agg['std'].fillna(0)

    ax.plot(agg[x_col], agg['mean'],
            color=PAL_METHOD['Vanilla'], linewidth=2.5, label='Vanilla mean')
    ax.fill_between(
        agg[x_col],
        (agg['mean'] - agg['std']).clip(lower=0),
        (agg['mean'] + agg['std']).clip(upper=1),
        alpha=0.2, color=PAL_METHOD['Vanilla']
    )

    g_vals = g_focus[col]
    if len(g_vals):
        gm, gstd = g_vals.mean(), g_vals.std()
        ax.axhline(gm, color=PAL_METHOD.get(greedy_label, '#00b894'),
                   linestyle='--', linewidth=2.2, label=f'{greedy_label} mean')
        ax.axhspan(max(0, gm - gstd), min(1, gm + gstd),
                   alpha=0.12, color=PAL_METHOD.get(greedy_label, '#00b894'))

    if greedy_ref_x is not None:
        ax.axvline(greedy_ref_x,
                   color=PAL_METHOD.get(greedy_label, '#00b894'),
                   linestyle=':', linewidth=1.8, alpha=0.7,
                   label=f'{greedy_label} ref')

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xlabel(x_label, fontsize=11)
    ax.set_ylabel(f'{metric} (Broad)', fontsize=11)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)


cp_focus = df_vanilla[df_vanilla['Alpha'] == FOCUS_ALPHA]
g_focus  = df_greedy[df_greedy['Alpha'] == FOCUS_ALPHA]

for x_col, x_label, greedy_ref in [
    ('Iteration',   'Iteration',      greedy_iterations(FOCUS_ALPHA, n_features)),
    ('Elapsed_Sec', 'Time (seconds)', g_focus['Elapsed_Sec'].mean() if len(g_focus) else None),
]:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, metric in enumerate(METRICS):
        _band_axes(axes[i], x_col, metric, cp_focus, g_focus, greedy_ref, x_label)

    fig.suptitle(
        f'Mean ± 1 Std (Broad GT) vs {x_label} — Alpha = {FOCUS_ALPHA}\n{DATASET} | {TARGET_ESTIMATOR}',
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

## 16. Key Findings

In [ ]:
print('=' * 80)
print('KEY FINDINGS — SYNTHETIC GROUND TRUTH ANALYSIS (Broad GT)')
print(f'GT definition: informative + redundant features (n={gt_broad} out of {n_features})')
print('=' * 80)

for metric in METRICS:
    col = gt_col(metric)
    print(f'\n  {metric}:')
    for alpha in alphas:
        v = df_vanilla_final[df_vanilla_final['Alpha'] == alpha][col]
        g = df_greedy_final[ df_greedy_final['Alpha']  == alpha][col]
        print(f'    alpha={alpha:6g} | Vanilla: {v.mean():.3f} ± {v.std():.3f}  |  '
              f'{greedy_label}: {g.mean():.3f} ± {g.std():.3f}')

print(f'\n--- CONVERGENCE (Vanilla — iteration where mean Recall/F1 first exceeds threshold) ---')
for threshold, metric in [(0.80, 'Recall'), (0.90, 'Recall'), (0.90, 'F1')]:
    print(f'\n  {metric} >= {threshold}:')
    for alpha in alphas:
        cp = df_vanilla[df_vanilla['Alpha'] == alpha]
        agg = cp.groupby('Iteration')[gt_col(metric)].mean()
        above = agg[agg >= threshold]
        if len(above):
            print(f'    alpha={alpha:6g} | First at iteration {above.index[0]}')
        else:
            print(f'    alpha={alpha:6g} | Never reached {threshold}')

print('\n' + '=' * 80)

## 17. Export

In [ ]:
df_tests.to_csv('data/synthetic_gt_statistical_tests.csv', index=False)
df_final_summary.to_csv('data/synthetic_gt_final_summary.csv', index=False)
print('Exported:')
print('  data/synthetic_gt_statistical_tests.csv')
print('  data/synthetic_gt_final_summary.csv')